In [1]:
# 01_data_pipeline.ipynb
# Collect and store price data + macro data

import yfinance as yf
import pandas as pd
import numpy as np
import pandas_datareader as pdr
from dotenv import load_dotenv
import os

env_path = os.path.expanduser("~/Desktop/SentriVaR-500/.env")
load_dotenv(env_path)
NEWSAPI_KEY = os.getenv("NEWSAPI_KEY")

START_DATE = "2018-01-01"
TICKERS = ["AAPL", "MSFT", "GOOGL", "JPM", "SOXX"]

print(f"API key check: {NEWSAPI_KEY[:8]}...")
print(f"Tickers: {TICKERS}")

API key check: 243b95d4...
Tickers: ['AAPL', 'MSFT', 'GOOGL', 'JPM', 'SOXX']


In [2]:
# Fetch price data
df = yf.download(TICKERS, start=START_DATE, auto_adjust=True, progress=False)["Close"]
df = df.dropna()

print(f"Data shape: {df.shape}")
print(f"Date range: {df.index[0].date()} ~ {df.index[-1].date()}")
df.tail()

Data shape: (2141, 5)
Date range: 2018-01-02 ~ 2026-07-10


Ticker,AAPL,GOOGL,JPM,MSFT,SOXX
Date,,,,,
2026-07-06,312.660004,366.459991,337.720001,386.739990,581.510010
2026-07-07,310.660004,367.029999,339.220001,388.839996,551.690002
2026-07-08,313.390015,361.920013,330.619995,383.339996,562.030029
2026-07-09,316.220001,358.890015,335.470001,384.359985,581.700012
2026-07-10,315.320007,357.179993,336.470001,385.100006,581.340027


In [3]:
# Fetch macro data
vix = pdr.get_data_fred("VIXCLS", start=START_DATE)
spread = pdr.get_data_fred("T10Y2Y", start=START_DATE)
cpi = pdr.get_data_fred("CPIAUCSL", start=START_DATE)

macro = pd.concat([vix, spread, cpi], axis=1)
macro.columns = ["VIX", "Spread", "CPI"]
macro = macro.dropna()

print(f"Macro data shape: {macro.shape}")
macro.tail()

Macro data shape: (63, 3)


/var/folders/sg/fpyw0z4d4qjbj_j45q9wv3y40000gn/T/ipykernel_63800/698314631.py:6: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  macro = pd.concat([vix, spread, cpi], axis=1)


,VIX,Spread,CPI
DATE,,,
2025-07-01,16.83,0.48,322.169
2025-08-01,20.38,0.54,323.291
2025-12-01,17.24,0.55,326.031
2026-04-01,24.54,0.52,332.407
2026-05-01,16.99,0.51,333.979


In [4]:
# Keep daily and monthly series separate — different frequencies
# VIX, Spread = daily / CPI = monthly
macro_daily = pd.concat([vix, spread], axis=1, sort=False)
macro_daily.columns = ["VIX", "Spread"]
macro_daily = macro_daily.dropna()

macro_monthly = cpi.copy()
macro_monthly.columns = ["CPI"]

print(f"Daily macro data: {macro_daily.shape}")
print(f"Monthly CPI data: {macro_monthly.shape}")
macro_daily.tail()# Create data folder if it doesn't exist, then save


Daily macro data: (2126, 2)
Monthly CPI data: (101, 1)


,VIX,Spread
DATE,,
2026-07-02,16.15,0.35
2026-07-06,15.57,0.35
2026-07-07,16.13,0.36
2026-07-08,16.90,0.35
2026-07-09,15.84,0.38


In [5]:
# Create data folder if it doesn't exist, then save
import os

data_path = os.path.expanduser("~/Desktop/SentriVaR-500/data")
os.makedirs(data_path, exist_ok=True)

df.to_csv(f"{data_path}/prices.csv")
macro_daily.to_csv(f"{data_path}/macro_daily.csv")
macro_monthly.to_csv(f"{data_path}/macro_monthly.csv")

print("Saved:")
print(f"  prices.csv        — {df.shape}")
print(f"  macro_daily.csv   — {macro_daily.shape}")
print(f"  macro_monthly.csv — {macro_monthly.shape}")

Saved:
  prices.csv        — (2141, 5)
  macro_daily.csv   — (2126, 2)
  macro_monthly.csv — (101, 1)
